### Exploratory Data Analysis

### Day-Level Aggregation

Pivots the raw EAV data into one row per (patient, day).  
Produces the daily feature matrix that Task 1C will use as input for the windowed instance dataset,  
and reports quality metrics to assess how many usable instances are available for modeling.

In [2]:
import pandas as pd
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────
DATA_PATH = "../data/raw/dataset_mood_smartphone.csv"
ID_COL   = "id"
TIME_COL = "time"
VAR_COL  = "variable"
VAL_COL  = "value"

# Aggregation rules per variable
MEAN_VARS  = ["mood", "circumplex.arousal", "circumplex.valence", "activity"]
SUM_VARS   = [
    "screen", "call", "sms",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
    "appCat.social", "appCat.travel", "appCat.unknown",
    "appCat.utilities", "appCat.weather",
]

# Variables required to call a day "fully observed" for modeling
MAJOR_PREDICTORS = ["activity", "screen", "circumplex.arousal", "circumplex.valence"]

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df["date"] = df[TIME_COL].dt.normalize()
df[VAL_COL] = pd.to_numeric(df[VAL_COL], errors="coerce")
print(f"Loaded {len(df):,} records")

Loaded 376,912 records


#### **1. Pivot to patient-day level**

In [4]:
# ── Mean aggregations ─────────────────────────────────────────────────────
mean_pivot = (
    df[df[VAR_COL].isin(MEAN_VARS)]
    .groupby([ID_COL, "date", VAR_COL])[VAL_COL]
    .mean()
    .unstack(VAR_COL)
)
mean_pivot.columns = [f"{c}_mean" for c in mean_pivot.columns]

# ── Sum aggregations ──────────────────────────────────────────────────────
sum_pivot = (
    df[df[VAR_COL].isin(SUM_VARS)]
    .groupby([ID_COL, "date", VAR_COL])[VAL_COL]
    .sum()
    .unstack(VAR_COL)
)
sum_pivot.columns = [f"{c}_sum" for c in sum_pivot.columns]

# ── Number of raw measurements that day ──────────────────────────────────
n_measurements = (
    df.groupby([ID_COL, "date"])
    .size()
    .rename("n_measurements")
)

# ── Combine ───────────────────────────────────────────────────────────────
day_df = (
    mean_pivot
    .join(sum_pivot, how="outer")
    .join(n_measurements, how="left")
    .reset_index()
)

# Convenience flag: does mood exist this day?
day_df["mood_available"] = day_df["mood_mean"].notna().astype(int)

print(f"Patient-days: {len(day_df):,}")
day_df.head()

Patient-days: 1,973


,id,date,activity_mean,circumplex.arousal_mean,circumplex.valence_mean,mood_mean,appCat.builtin_sum,appCat.communication_sum,appCat.entertainment_sum,appCat.finance_sum,...,appCat.social_sum,appCat.travel_sum,appCat.unknown_sum,appCat.utilities_sum,appCat.weather_sum,call_sum,screen_sum,sms_sum,n_measurements,mood_available
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2,0
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1,0
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,7.0,NaN,2.0,9,0
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.0,NaN,3.0,5,0
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1,0


#### **2. Quality metrics at day level**

In [5]:
n_total_days = len(day_df)

# Usable days = days that have at least a mood record (target exists)
n_with_target = day_df["mood_available"].sum()
pct_with_target = n_with_target / n_total_days * 100

# Days where all major predictors are non-null
major_cols = [f"{v}_mean" if v in MEAN_VARS else f"{v}_sum" for v in MAJOR_PREDICTORS]
major_cols = [c for c in major_cols if c in day_df.columns]   # guard missing vars

n_fully_observed = day_df[major_cols].notna().all(axis=1).sum()
pct_fully_observed = n_fully_observed / n_total_days * 100

# Days with BOTH target and all major predictors
n_usable = (
    day_df[day_df["mood_available"] == 1][major_cols]
    .notna().all(axis=1).sum()
)
pct_usable = n_usable / n_total_days * 100

# Average number of distinct variables observed per day
all_feature_cols = [c for c in day_df.columns
                    if c not in [ID_COL, "date", "mood_available", "n_measurements"]]
day_df["n_active_vars"] = day_df[all_feature_cols].notna().sum(axis=1)
avg_active_vars = day_df["n_active_vars"].mean()

print("=" * 52)
print("DAY-LEVEL QUALITY METRICS")
print("=" * 52)
print(f"  Total patient-days                  : {n_total_days:,}")
print(f"  Days with mood (target available)   : {n_with_target:,}  ({pct_with_target:.1f}%)")
print(f"  Days with all major predictors      : {n_fully_observed:,}  ({pct_fully_observed:.1f}%)")
print(f"  Days usable (target + predictors)   : {n_usable:,}  ({pct_usable:.1f}%)")
print(f"  Avg active variables per day        : {avg_active_vars:.1f}")
print("=" * 52)

DAY-LEVEL QUALITY METRICS
  Total patient-days                  : 1,973
  Days with mood (target available)   : 1,268  (64.3%)
  Days with all major predictors      : 1,133  (57.4%)
  Days usable (target + predictors)   : 1,133  (57.4%)
  Avg active variables per day        : 7.9


In [6]:
# ── Per-column missingness at day level ───────────────────────────────────
# Useful for spotting which features will need imputation at this granularity
miss = (
    day_df[all_feature_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_%")
    .sort_values(ascending=False)
    .to_frame()
)
print("\nMissingness per feature at day level:")
miss


Missingness per feature at day level:


,missing_%
appCat.weather_sum,94.27
appCat.game_sum,90.22
appCat.finance_sum,89.51
appCat.unknown_sum,86.62
appCat.office_sum,86.01
appCat.travel_sum,78.26
appCat.utilities_sum,78.05
sms_sum,62.80
appCat.entertainment_sum,56.92
appCat.social_sum,49.92


#### **3. Full display and export**

In [7]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
day_df.head(10)

,id,date,activity_mean,circumplex.arousal_mean,circumplex.valence_mean,mood_mean,appCat.builtin_sum,appCat.communication_sum,appCat.entertainment_sum,appCat.finance_sum,appCat.game_sum,appCat.office_sum,appCat.other_sum,appCat.social_sum,appCat.travel_sum,appCat.unknown_sum,appCat.utilities_sum,appCat.weather_sum,call_sum,screen_sum,sms_sum,n_measurements,mood_available,n_active_vars
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,NaN,NaN,2,0,1
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,NaN,1,0,1
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.000,NaN,2.000,9,0,2
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,NaN,3.000,5,0,2
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,1,0,1
5,AS14.01,2014-02-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,NaN,1.000,3,0,2
6,AS14.01,2014-02-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000,NaN,NaN,3,0,1
7,AS14.01,2014-02-26,NaN,-0.250,0.750,6.250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,2.000,15,1,5
8,AS14.01,2014-02-27,NaN,0.000,0.333,6.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,1,3
9,AS14.01,2014-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000,NaN,NaN,4,0,1


In [8]:
day_df.to_csv("../outputs/daily_aggregated.csv", index=False)
miss.to_csv("../outputs/daily_missingness.csv")
print("Saved daily_aggregated.csv and daily_missingness.csv")

Saved daily_aggregated.csv and daily_missingness.csv


#### Why this matters


#### Notes 

**Key decisions to be aware of**:


**`daily_aggregated.csv`**: this file is the direct input to Task 1C. We won't need to re-pivot the raw data again; just load this and apply our sliding window on top of it.



#### Additional Analysis 

**455 days have only 1 active variable, and it's almost always call or sms.** call_sum accounts for 366 of these, sms_sum for 87. These are days where the phone logged one event but nothing else; no mood, no screen, no app usage. These rows are essentially uninformative as training instances and will be naturally excluded once we filter for days with target + predictors. Worth flagging explicitly.


**n_active_vars max is 18, not 19.** No single day has all 19 variables recorded simultaneously for any patient. This is consistent with the extreme sparsity of variables like appCat.weather and appCat.game, they virtually never co-occur with everything else on the same day.
